In [ ]:
# %% Deep learning - Section 21.201
#    CIFAR10 with autoencoder-pretrained model

# This code pertains a deep learning course provided by Mike X. Cohen on Udemy:
#   > https://www.udemy.com/course/deeplearning_x
# The "base" code in this repository is adapted (with very minor modifications)
# from code developed by the course instructor (Mike X. Cohen), while the
# "exercises" and the "code challenges" contain more original solutions and
# creative input from my side. If you are interested in DL (and if you are
# reading this statement, chances are that you are), go check out the course, it
# is singularly good.

In [2]:
# %% Libraries and modules
import numpy                  as np
import matplotlib.pyplot      as plt
import torch
import torch.nn               as nn
import seaborn                as sns
import copy
import torch.nn.functional    as F
import pandas                 as pd
import scipy.stats            as stats
import sklearn.metrics        as skm
import time
import sys
import imageio.v2             as imageio
import torchvision
import torchvision.transforms as T

from torch.utils.data                 import DataLoader,TensorDataset,Dataset,Subset
from sklearn.model_selection          import train_test_split
from google.colab                     import files
from torchsummary                     import summary
from scipy.stats                      import zscore
from sklearn.decomposition            import PCA
from scipy.signal                     import convolve2d
from torchsummary                     import summary
from IPython                          import display
from matplotlib_inline.backend_inline import set_matplotlib_formats
set_matplotlib_formats('svg')
plt.style.use('default')


In [ ]:
# %% Use GPU

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)


In [41]:
# %% CIFAR10 data

# Transforms
transform = T.Compose([ T.ToTensor(),
                        T.Normalize([.5,.5,.5],[.5,.5,.5])
                       ])

# Import data and apply the transform
train_set = torchvision.datasets.CIFAR10(root='./data', train=True,  download=True, transform=transform)
test_set  = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

# Convert into DataLoader objects
batchsize        = 32
train_loader_full = DataLoader(train_set,batch_size=batchsize,shuffle=True,drop_last=True)
test_loader_full  = DataLoader(test_set,batch_size=256)


In [42]:
# %% Smaller sets

# Create also smaller DataLoaders with only 2k images
train_set_small    = torch.utils.data.Subset(train_set,range(2000))
train_loader_small = DataLoader(train_set_small,batch_size=batchsize,shuffle=True)

test_set_small    = torch.utils.data.Subset(test_set,range(2000))
test_loader_small = DataLoader(test_set_small,batch_size=batchsize,shuffle=True)


In [ ]:
# %% inspect a few random images

X,y = next(iter(train_loader_small))

# Plot
phi = (1 + np.sqrt(5)) / 2
fig,axs = plt.subplots(4,4,figsize=(phi*6,6))

for (i,ax) in enumerate(axs.flatten()):

    # Get image and undo normalisation (transpose back to 32x32x3)
    pic = X.data[i].numpy().transpose((1,2,0))
    pic = pic/2 + .5

    # Label
    label = train_set.classes[y[i]]

    ax.imshow(pic)
    ax.text(14,0,label,ha='center',fontweight='bold',color='k',backgroundcolor='y')
    ax.axis('off')

plt.tight_layout()

plt.savefig('figure33_cifar10_autoencoder_pretraining.png')
plt.show()
files.download('figure33_cifar10_autoencoder_pretraining.png')


In [7]:
# %% Function to generate the autencoder model

def gen_ae_model(printtoggle=False):

    class aenet(nn.Module):
        def __init__(self,printtoggle):
            super().__init__()

            self.print = printtoggle

            # Encoder layers (stride instead of pool to downsample)
            self.encconv1  = nn.Conv2d( 3,16,4,padding=1,stride=2) # output size: (32+2*1-4)/2 + 1 = 16
            self.encconv2  = nn.Conv2d(16,32,4,padding=1,stride=2) # output size: (16+2*1-4)/2 + 1 = 8
            self.encconv3  = nn.Conv2d(32,64,4,padding=1,stride=2) # output size: (8 +2*1-4)/2 + 1 = 4

            # Decoder layers
            self.decconv1  = nn.ConvTranspose2d(64,32,4,padding=1,stride=2)
            self.decconv2  = nn.ConvTranspose2d(32,16,4,padding=1,stride=2)
            self.decconv3  = nn.ConvTranspose2d(16, 3,4,padding=1,stride=2)

        def forward(self,x):

            if self.print: print(f'Input: {list(x.shape)}')

            # Encoder block
            x = F.leaky_relu( self.encconv1(x) )
            if self.print: print(f'First encoder block: {list(x.shape)}')

            x = F.leaky_relu( self.encconv2(x) )
            if self.print: print(f'Second encoder block: {list(x.shape)}')

            x = F.leaky_relu( self.encconv3(x) )
            if self.print: print(f'Third encoder block: {list(x.shape)}')

            # Decoder block
            x = F.leaky_relu( self.decconv1(x) )
            if self.print: print(f'First decoder block: {list(x.shape)}')

            x = F.leaky_relu( self.decconv2(x) )
            if self.print: print(f'Second decoder block: {list(x.shape)}')

            x = F.leaky_relu( self.decconv3(x) )
            if self.print: print(f'Decoder output: {list(x.shape)}')

            return x

    # Create model instance
    net = aenet(printtoggle)

    # Loss function
    lossfun = nn.MSELoss()

    # Optimizer
    optimizer = torch.optim.Adam(net.parameters(),lr=.001,weight_decay=1e-5)

    return net,lossfun,optimizer


In [ ]:
# %% Test the model on one batch

tmp_net,loss_fun,optimizer = gen_ae_model(True)

X,y  = next(iter(train_loader_small))
yHat = tmp_net(X)

# Check sizes of output
print('\nOutput size:')
print(yHat.shape)

# Check loss
loss = loss_fun(yHat,X)
print(' ')
print('Loss:')
print(loss)


In [13]:
# %% Function to train the autoencoder model

def train_ae_model(ae_net,loss_fun,optimizer):

    # Parameters, inizialise vars
    numepochs = 15

    # Ship to GPU
    ae_net.to(device)

    # Preallocate losses
    train_loss = torch.zeros(numepochs)
    dev_loss   = torch.zeros(numepochs)

    # Loop over epochs
    for epochi in range(numepochs):

        batch_loss = []

        for X,y in train_loader_full:

            # Ship data to GPU
            X = X.to(device)
            y = y.to(device)

            # Forward propagation and loss
            yHat = ae_net(X)
            loss = loss_fun(yHat,X)

            # Backpropagation
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            # Loss from this batch
            batch_loss.append(loss.item())

        # Average losses across batches
        train_loss[epochi] = np.mean(batch_loss)

        # Dev performance
        ae_net.eval()
        X,y = next(iter(test_loader_full))

        # Ship data to GPU
        X = X.to(device)
        y = y.to(device)

        # Forward propagation and loss
        with torch.no_grad():
            yHat = ae_net(X)
            loss = loss_fun(yHat,X)

        # Dev loss (one batch)
        dev_loss[epochi] = loss.item()
        ae_net.train()

    return train_loss,dev_loss,ae_net


In [14]:
# %% Fit the AE model

# Takes ~4 mins on GPU
ae_net,loss_fun,optimizer  = gen_ae_model()
train_loss,dev_loss,ae_net = train_ae_model(ae_net,loss_fun,optimizer)


In [ ]:
# %% Plotting

phi = (1 + np.sqrt(5)) / 2
fig = plt.figure(figsize=(phi*5,5))

plt.plot(train_loss,'s-',label='Train')
plt.plot(dev_loss,'o-',label='Dev')
plt.xlabel('Epochs')
plt.ylabel('Loss (MSE)')
plt.title('Model loss')
plt.legend()

plt.savefig('figure34_cifar10_autoencoder_pretraining.png')
plt.show()
files.download('figure34_cifar10_autoencoder_pretraining.png')


In [ ]:
# %% Plotting

# Get some data
X,y = next(iter(test_loader_full))

# Forward propagation and loss
ae_net.cpu()
ae_net.eval()
yHat = ae_net(X)

phi = (1 + np.sqrt(5)) / 2
fig,axs = plt.subplots(2,10,figsize=(2*phi*5,5))

for i in range(10):

    # Predicted (undo normalisation and reshape)
    pic = yHat[i,:,:,:].detach().numpy().transpose((1,2,0))
    pic = pic/2 + .5
    axs[0,i].imshow(pic)
    axs[0,i].axis('off')

    # Actual (undo normalisation)
    pic = X[i,:,:,:].detach().numpy().transpose((1,2,0))
    pic = pic/2 + .5
    axs[1,i].imshow(pic)
    axs[1,i].axis('off')

    if i==0:
        axs[0,0].text(-6,14,'Reconstructed',rotation=90,va='center')
        axs[1,0].text(-6,14,'Original',rotation=90,va='center')

plt.savefig('figure35_cifar10_autoencoder_pretraining.png')
plt.show()
files.download('figure35_cifar10_autoencoder_pretraining.png')


In [43]:
# %% Function to generate the classifier model

def gen_classifier_model(printtoggle=False):

    class CNN_net(nn.Module):
        def __init__(self,printtoggle):
            super().__init__()

            # Print toggle
            self.print = printtoggle

            # Encoder layers
            self.encconv1  = nn.Conv2d( 3,16,4,padding=1,stride=2) # output size: (28+2*1-4)/2 + 1 = 14
            self.encconv2  = nn.Conv2d(16,32,4,padding=1,stride=2) # output size: (14+2*1-4)/2 + 1 = 7
            self.encconv3  = nn.Conv2d(32,64,4,padding=1,stride=2) # output size: ( 7+2*1-4)/2 + 1 = 4

            # Linear layers
            self.fc1  = nn.Linear(4*4*64,128)
            self.fc2  = nn.Linear(128,    64)
            self.fc3  = nn.Linear(64,     10)

        def forward(self,x):

            if self.print: print(f'Input: {list(x.shape)}')

            # Encoder block
            x = F.leaky_relu(self.encconv1(x))
            if self.print: print(f'First encoder layer: {list(x.shape)}')

            x = F.leaky_relu(self.encconv2(x))
            if self.print: print(f'Second encoder layer: {list(x.shape)}')

            x = F.leaky_relu(self.encconv3(x))
            if self.print: print(f'Third encoder layer: {list(x.shape)}')

            # Vectorise
            n_units = x.shape.numel()/x.shape[0]
            x       = x.view(-1,int(n_units))
            if self.print: print(f'Post-convolution vectorised: {list(x.shape)}')

            # Linear layers
            x = F.leaky_relu(self.fc1(x))
            if self.print: print(f'First linear layer: {list(x.shape)}')

            x = F.leaky_relu(self.fc2(x))
            if self.print: print(f'Second linear layer: {list(x.shape)}')

            x = self.fc3(x)
            if self.print: print(f'Output linear layer: {list(x.shape)}')

            return x

    # Create model instance
    net = CNN_net(printtoggle)

    # Loss function
    loss_fun = nn.CrossEntropyLoss()

    # Optimizer
    optimizer = torch.optim.Adam(net.parameters(),lr=.001)

    return net,loss_fun,optimizer


In [ ]:
# %% Test the model on one batch

tmp_net,loss_fun,optimizer = gen_classifier_model(True)

X,y  = next(iter(train_loader_small))
yHat = tmp_net(X)

# Check sizes of output
print('\nOutput size:')
print(yHat.shape)

# Check loss
loss = loss_fun(yHat,y)
print()
print('Loss:')
print(loss)


In [23]:
# %% Function to train the classifier model

def train_classifier_model(net,loss_fun,optimizer,train_loader,test_loader):

    # Parameters
    numepochs = 10

    # Ship to GPU
    net.to(device)

    # Preallocate losses and accuracies
    train_loss = torch.zeros(numepochs)
    dev_loss   = torch.zeros(numepochs)
    train_acc  = torch.zeros(numepochs)
    dev_acc    = torch.zeros(numepochs)

    # Loop over epochs
    for epochi in range(numepochs):

        batch_loss = []
        batch_acc  = []

        for X,y in train_loader:

            # Ship data to GPU
            X = X.to(device)
            y = y.to(device)

            # Forward propagation and loss
            yHat = net(X)
            loss = loss_fun(yHat,y)

            # Backpropagation
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            # Loss and accuracy from this batch
            batch_loss.append(loss.item())
            batch_acc.append(torch.mean((torch.argmax(yHat,axis=1) == y).float()).item())

        # Average losses across batches
        train_loss[epochi] = np.mean(batch_loss)
        train_acc[epochi]  = 100*np.mean(batch_acc)

        # Dev performance
        net.eval()
        X,y = next(iter(test_loader))

        # Ship data to GPU
        X = X.to(device)
        y = y.to(device)

        # Forward propagation and loss
        with torch.no_grad():
            yHat = net(X)
            loss = loss_fun(yHat,y)

        # Dev loss (one batch)
        dev_loss[epochi] = loss.item()
        dev_acc[epochi]  = 100 * torch.mean((torch.argmax(yHat,axis=1) == y).float()).item()

        net.train()

    return train_acc,dev_acc,train_loss,dev_loss,net


In [58]:
# %% Train a new model from scratch

# Create a 'naive' classifier network (trained from scratch), takes ~3 mins full
naive_net,loss_fun,optimizer = gen_classifier_model()
train_acc_naive,dev_acc_naive,train_loss_naive,dev_loss_naive,naive_net = train_classifier_model(naive_net,loss_fun,optimizer,train_loader_full,test_loader_full)


In [ ]:
# %% Plotting

phi = (1 + np.sqrt(5)) / 2
fig,ax = plt.subplots(1,2,figsize=(1.5*phi*6,6))

ax[0].plot(train_loss_naive,'s-',label='Train')
ax[0].plot(dev_loss_naive,'o-',label='Dev')
ax[0].set_xlabel('Epochs')
ax[0].set_ylabel('Loss (MSE)')
ax[0].set_title('Model loss')

ax[1].plot(train_acc_naive,'s-',label='Train')
ax[1].plot(dev_acc_naive,'o-',label='Dev')
ax[1].set_xlabel('Epochs')
ax[1].set_ylabel('Accuracy (%)')
ax[1].set_title(f'Final model test accuracy: {dev_acc_naive[-1]:.2f}%')
ax[1].legend()

plt.savefig('figure36_cifar10_autoencoder_pretraining.png')
plt.show()
files.download('figure36_cifar10_autoencoder_pretraining.png')


In [ ]:
# %% Build pretrained model

pretrained_net,loss_fun,optimizer = gen_classifier_model()

# Note about the code below :
# Both networks have the same number of layers overall; in other applications
# you may need to modify the code to find the matching layers

# Replace the conv. weights in target model with encoder weights from source model
for target,source in zip(pretrained_net.named_parameters(),ae_net.named_parameters()):
    print('Pretrained: ' + target[0] + '  --  AE_net: ' + source[0])
    if 'enc' in target[0]:
        # Copy and freeze convolutional layers
        target[1].data = copy.deepcopy( source[1].data )
        # target[1].requires_grad = False

# Double-check matching
print(pretrained_net.cpu().encconv1.weight[10] - ae_net.encconv1.weight[10])


In [61]:
# %% Fine-tune the pretrained model

# Tuning, takes ~3 mins full
train_acc_pre,dev_acc_pre,train_loss_pre,dev_loss_pre,pretrained_net = train_classifier_model(pretrained_net,loss_fun,optimizer,train_loader_full,test_loader_full)


In [ ]:
# %% Plotting

phi = (1 + np.sqrt(5)) / 2
fig,ax = plt.subplots(1,2,figsize=(1.5*phi*6,6))

ax[0].plot(train_loss_pre,'s-',color='tab:red',label='Train (pretrained)')
ax[0].plot(dev_loss_pre,'o--',color='tab:red',label='Dev (pretrained)',alpha=0.75)
ax[0].plot(train_loss_naive,'s-',color='tab:blue',label='Train (naïve)')
ax[0].plot(dev_loss_naive,'o--',color='tab:blue',label='Dev (naïve)',alpha=0.75)
ax[0].set_xlabel('Epochs')
ax[0].set_ylabel('Loss (MSE)')
ax[0].set_title('Model loss')
ax[0].legend()

ax[1].plot(train_acc_pre,'s-',color='tab:red',label='Train (pretrained)')
ax[1].plot(dev_acc_pre,'o--',color='tab:red',label='Dev (pretrained)',alpha=0.75)
ax[1].plot(train_acc_naive,'s-',color='tab:blue',label='Train (naïve)')
ax[1].plot(dev_acc_naive,'o--',color='tab:blue',label='Dev (naïve)',alpha=0.75)
ax[1].set_xlabel('Epochs')
ax[1].set_ylabel('Accuracy (%)')
ax[1].set_title(f'Final test [naïve, pretrained] accuracy: [{dev_acc_naive[-1]:.2f}%, {dev_acc_pre[-1]:.2f}%]')
ax[1].legend()

plt.savefig('figure37_cifar10_autoencoder_pretraining.png')
plt.show()
files.download('figure37_cifar10_autoencoder_pretraining.png')


In [ ]:
# %% Exercise 1
#    Performance was overall low. But we only trained on 2k images, whereas the full CIFAR10 dataset has 60,000 images.
#    Maybe the benefit of AE-pretraining will be seen with a larger image size? Modify the code to use the entire dataset.

# Indeed the performance improves and become more stable


In [56]:
# %% Exercise 2
#    You discovered in the "CNN Milestone" section (project 1) that a simple classifier doesn't do very well on this
#    dataset, and that we got better performance from a more complex model. Modify the classifier here so that it matches
#    the architecture from that project. Does the AE-pretraining help with that model architecture?

# Adding batchnorm and dropout to the fc layers does indeed help a bit

# Modified classifier part
def gen_classifier_model(printtoggle=False):

    class CNN_net(nn.Module):
        def __init__(self,printtoggle):
            super().__init__()

            self.print = printtoggle

            self.encconv1  = nn.Conv2d( 3,16,4,padding=1,stride=2)
            self.encconv2  = nn.Conv2d(16,32,4,padding=1,stride=2)
            self.encconv3  = nn.Conv2d(32,64,4,padding=1,stride=2)

            self.fc1       = nn.Linear(4*4*64,128)
            self.bnorm_fc1 = nn.BatchNorm1d(128)
            self.drop_fc1  = nn.Dropout(p=0.5)

            self.fc2       = nn.Linear(128,64)
            self.bnorm_fc2 = nn.BatchNorm1d(64)
            self.drop_fc2  = nn.Dropout(p=0.5)

            self.fc3  = nn.Linear(64,     10)

        def forward(self,x):

            if self.print: print(f'Input: {list(x.shape)}')

            x = F.leaky_relu(self.encconv1(x))
            if self.print: print(f'First encoder layer: {list(x.shape)}')

            x = F.leaky_relu(self.encconv2(x))
            if self.print: print(f'Second encoder layer: {list(x.shape)}')

            x = F.leaky_relu(self.encconv3(x))
            if self.print: print(f'Third encoder layer: {list(x.shape)}')

            n_units = x.shape.numel()/x.shape[0]
            x       = x.view(-1,int(n_units))
            if self.print: print(f'Post-convolution vectorised: {list(x.shape)}')

            x = F.leaky_relu(self.fc1(x))
            x = self.bnorm_fc1(x)
            if self.print: print(f'First linear layer: {list(x.shape)}')

            x = F.leaky_relu(self.fc2(x))
            x = self.bnorm_fc2(x)
            if self.print: print(f'Second linear layer: {list(x.shape)}')

            x = self.fc3(x)
            if self.print: print(f'Output linear layer: {list(x.shape)}')

            return x

    net       = CNN_net(printtoggle)
    loss_fun  = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(net.parameters(),lr=.001)

    return net,loss_fun,optimizer
